# 03 | WorldCereal 2026: classification and evaluation
Train the same three fixed classifiers as Notebook 01 using the newly generated seasonal embeddings. Use identical held-out reference groups and report performance on the retained sample subset. Differences may reflect encoder, pooling and temporal selection changes together; this is not an isolated checkpoint ablation.

## 1. Load saved features and check experiment identity
No GPU is required. Keep the 2025 baseline untouched. Use the new feature manifest to prevent mismatched model inference.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
drive.mount('/content/drive')
root=Path('/content/drive/MyDrive/ESA_EO_Projects')
matches=[p.parent.parent for p in root.rglob('extractions.parquet') if (p.parent/'processed_training_df.parquet').is_file()]
assert len(matches)==1, matches
PROJECT=matches[0]; DATA=PROJECT/'data'; BASE=PROJECT/'outputs'; OLD=PROJECT/'models'
OUT=BASE/'2026'; MODELS=OLD/'2026'
for d in [OUT,MODELS]: d.mkdir(parents=True,exist_ok=True)
print(PROJECT)
import numpy as np,pandas as pd, matplotlib.pyplot as plt,seaborn as sns,joblib,time,sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score,classification_report,ConfusionMatrixDisplay
m=json.loads((MODELS/'embedding_manifest.json').read_text()); F=m['features']
df=pd.read_parquet(OUT/'reference_embeddings.parquet'); assert df.sample_id.is_unique and np.isfinite(df[F]).all().all()
train=df[df.split=='train']; test=df[df.split=='test']; assert set(train.ref_id).isdisjoint(test.ref_id)
assert set(train.downstream_class)==set(test.downstream_class)
print('Train:',len(train),'Test:',len(test)); display(pd.crosstab(df.downstream_class,df.split))

## 2. Train fixed baseline and comparable classifiers
Keep the same model settings and seed. The test set is for evaluation, not hyperparameter selection. Random Forest remains the primary model.

In [ ]:
Xtr=train[F]; Xte=test[F]; ytr=train.downstream_class; yte=test.downstream_class
models={'majority_baseline':DummyClassifier(strategy='most_frequent'),'random_forest':make_pipeline(SimpleImputer(strategy='median'),RandomForestClassifier(n_estimators=150,min_samples_leaf=2,class_weight='balanced_subsample',n_jobs=2,random_state=42)),'hist_gradient_boosting':HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=31,learning_rate=.1,random_state=42)}
def metrics(y,p): return {'accuracy':accuracy_score(y,p),'balanced_accuracy':balanced_accuracy_score(y,p),'macro_f1':f1_score(y,p,average='macro',zero_division=0)}
rows=[]; predictions={}
for name,model in models.items():
    start=time.perf_counter(); model.fit(Xtr,ytr); p=model.predict(Xte); predictions[name]=p
    rows.append({'model':name,**metrics(yte,p),'seconds':time.perf_counter()-start})
results=pd.DataFrame(rows).set_index('model'); display(results.round(4)); results.to_csv(OUT/'model_comparison.csv')

## 3. Compare on exactly matched withheld samples
The 2025 baseline was trained on a larger or equal set. Join its saved Random Forest test predictions by sample ID and source, then compare both models only where both have predictions. This prevents a misleading comparison caused by test sample exclusions. The full original scores remain 80.83% accuracy and 65.89% macro F1.

In [ ]:
p=predictions['random_forest']; errors=test[['ref_id','sample_id','downstream_class']].copy(); errors['prediction']=p
old=pd.read_csv(BASE/'test_predictions.csv',dtype={'sample_id':str,'ref_id':str})
errors['sample_id']=errors.sample_id.astype(str); errors['ref_id']=errors.ref_id.astype(str)
paired=errors.merge(old,on=['ref_id','sample_id'],suffixes=('_new','_old'),validate='one_to_one')
assert len(paired)>0 and (paired.downstream_class_new==paired.downstream_class_old).all()
comparison=pd.DataFrame([{'model':'2025 RF matched',**metrics(paired.downstream_class_new,paired.prediction_old)},{'model':'2026 RF matched',**metrics(paired.downstream_class_new,paired.prediction_new)}]).set_index('model')
print('Matched withheld samples:',len(paired),'of',len(test)); display(comparison.round(4)); comparison.to_csv(OUT/'paired_baseline_comparison.csv')
ax=comparison.plot.bar(figsize=(8,4),ylim=(0,1)); ax.set_ylabel('Score'); plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(OUT/'paired_comparison.png',dpi=180); plt.show()

## 4. Diagnose per-class and source performance
Inspect minority class recall and confusion between crop groups. Source-level performance is not evidence of geographic independence. Preserve the preselected RF rather than choosing a winner based solely on test metrics.

In [ ]:
labels=sorted(yte.unique()); report=pd.DataFrame(classification_report(yte,p,labels=labels,output_dict=True,zero_division=0)).T
display(report.round(3)); report.to_csv(OUT/'classification_report.csv')
fig,ax=plt.subplots(figsize=(9,8)); ConfusionMatrixDisplay.from_predictions(yte,p,labels=labels,normalize='true',xticks_rotation=90,ax=ax,cmap='Blues',values_format='.2f'); plt.tight_layout(); plt.savefig(OUT/'confusion_matrix.png',dpi=180); plt.show()
errors['correct']=errors.downstream_class==errors.prediction; display(errors.groupby('ref_id').correct.agg(['size','mean'])); errors.to_csv(OUT/'test_predictions.csv',index=False)

## 5. Save the new classifier
Record the same feature ordering and checkpoint fingerprint as Notebook 02. Only load locally created model files. The mapping notebook refuses incompatible encoders.

In [ ]:
joblib.dump(models['random_forest'],MODELS/'random_forest_2026.joblib')
m['sklearn_version']=sklearn.__version__; m['classes']=models['random_forest'].classes_.tolist(); m['train_sources']=sorted(train.ref_id.unique()); m['test_sources']=sorted(test.ref_id.unique())
(MODELS/'classifier_manifest.json').write_text(json.dumps(m,indent=2)); print('Saved classifier and provenance')